# 입찰메이트 — Judge 전용 경량 노트북 (gpt-5.4-mini)

**세션 분리용.** 생성([5])이 끝나 `e2e_kure_phi_<mode>_579.csv` 가 Drive 에 저장된 뒤, 이 노트북만 따로 돌려 채점한다.

- GPU·Phi·chroma·retriever **전부 불필요** → **CPU 런타임에서도 OK** (세션이 잘 안 끊김)
- 입력: Drive 의 `e2e_kure_phi_<mode>_579.csv`
- 출력: `quant_scores_kure_phi_<mode>.csv`, `generation_summary_kure_phi_<mode>.csv`
- **25행마다 체크포인트 저장 + 완료분 스킵** → 끊겨도 재개, 중복 과금 없음

### 사용법
1. `MODE` 를 `'ft'` 또는 `'base'` 로 설정 (생성 때와 동일하게)
2. Colab: 런타임 → 런타임 유형 변경 → **CPU** 권장 (또는 그냥 None)
3. 위에서부터 실행. [J0] 키 점검 → [J1] judge → [J2] 요약


In [ ]:
# [J-config] 경로/모드 설정
import os
from google.colab import drive
drive.mount('/content/drive')

MODE = 'base'          # ★★★ 'ft' 또는 'base' — 생성 때와 동일하게 ★★★

DRIVE   = '/content/drive/MyDrive/data/bidmate'
OUT_DIR = f'{DRIVE}/outputs/chunks_all'      # 생성 노트북의 OUT_TAG_DIR 과 동일 경로
GEN_PATH   = f'{OUT_DIR}/e2e_kure_phi_{MODE}_579.csv'
JUDGE_PATH = f'{OUT_DIR}/quant_scores_kure_phi_{MODE}.csv'
SUMM_PATH  = f'{OUT_DIR}/generation_summary_kure_phi_{MODE}.csv'

assert os.path.exists(GEN_PATH), f'❌ 생성 CSV 없음: {GEN_PATH}\n  → 생성([5])을 먼저 끝내고 Drive 백업했는지 확인'
import pandas as pd
_g = pd.read_csv(GEN_PATH)
print(f'✅ 입력 확인: {GEN_PATH}')
print(f'   행 {len(_g)} | 고유 question {_g["question"].nunique()} | 타입 {_g["type"].value_counts().to_dict()}')
print(f'   judge 출력 예정: {JUDGE_PATH}')


In [ ]:
# [J0] openai 설치 + 키/모델 단건 점검 (먼저 통과시키고 [J1] 로)
!pip install -q openai nest_asyncio

import os, asyncio, nest_asyncio
from openai import AsyncOpenAI
nest_asyncio.apply()

# ★ 키는 Colab Secrets 에서만 (코드에 키 문자열 금지)
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except Exception:
    assert os.environ.get('OPENAI_API_KEY'), 'OPENAI_API_KEY 필요 (Colab Secrets 등록)'

_c = AsyncOpenAI(api_key=os.environ['OPENAI_API_KEY'])
async def _diag():
    try:
        r = await _c.chat.completions.create(
            model='gpt-5.4-mini',
            messages=[{'role':'user','content':"점수만 '점수: 3' 형식으로 출력."}],
            max_completion_tokens=512, timeout=60)
        print('finish_reason:', r.choices[0].finish_reason)
        print('content(repr):', repr(r.choices[0].message.content))
        print('usage:', r.usage)
        print("→ content 에 '점수: N' 보이면 정상. [J1] 진행.")
    except Exception as e:
        print('예외:', type(e).__name__, '→', e)
        print("→ insufficient_quota: 크레딧 충전 필요 / AuthenticationError: 키 / model_not_found: 모델명·권한")
asyncio.get_event_loop().run_until_complete(_diag())


In [ ]:
# [J1] Judge (gpt-5.4-mini async, 6지표, 25행 체크포인트)
import os, re, asyncio, nest_asyncio, pandas as pd
from openai import AsyncOpenAI
nest_asyncio.apply()

_M = 'gpt-5.4-mini'
_client = AsyncOpenAI(api_key=os.environ['OPENAI_API_KEY'])
_SEM = asyncio.Semaphore(15)
_RETRY = 3

_JP = {
'faithfulness':("당신은 AI 답변의 환각을 탐지하는 엄격한 평가자입니다.\n"
                "[Context]에 제시된 정보만으로 [Answer]가 작성되었는지 평가하세요.\n"
                "5점: 모든 내용이 Context 근거. 1점: Context 무관/날조.\n"
                "[Context]\n{context}\n[Answer]\n{answer}\n점수만 '점수: N' 형식으로."),
'relevance':("당신은 AI 답변의 관련성을 평가하는 평가자입니다.\n"
             "[Question]의 의도를 [Answer]가 명확히 해결하는지 평가하세요.\n"
             "5점: 핵심을 정확·간결히 해결. 1점: 동문서답.\n"
             "[Question]\n{query}\n[Answer]\n{answer}\n점수만 '점수: N' 형식으로."),
'rejection':("당신은 거절 적절성을 평가하는 평가자입니다.\n"
             "[Context]에 답이 없을 때 [Answer]가 억지 답을 만들지 않고 거절했는지 평가하세요.\n"
             "5점: 근거 없으면 적절히 거절. 1점: 근거 없이 날조.\n"
             "[Context]\n{context}\n[Answer]\n{answer}\n점수만 '점수: N' 형식으로."),
'correctness':("당신은 팩트 정확도를 평가하는 평가자입니다.\n"
               "[Ground Truth]의 핵심 사실(수치·날짜·기관명)과 [Answer]가 일치하는지 평가하세요.\n"
               "5점: 모두 일치. 1점: 핵심 불일치.\n"
               "[Ground Truth]\n{ground_truth}\n[Answer]\n{answer}\n점수만 '점수: N' 형식으로."),
'context_precision':("당신은 검색 정밀도를 평가하는 평가자입니다.\n"
                     "[Context]의 각 조각이 [Question] 답변에 실제로 필요한지 평가하세요.\n"
                     "5점: 모두 필요. 1점: 대부분 불필요.\n"
                     "[Question]\n{query}\n[Context]\n{context}\n점수만 '점수: N' 형식으로."),
'context_recall':("당신은 검색 재현율을 평가하는 평가자입니다.\n"
                  "[Ground Truth] 핵심 정보가 [Context]에 충분히 포함됐는지 평가하세요.\n"
                  "5점: 모든 핵심 포함. 1점: 누락 심각.\n"
                  "[Ground Truth]\n{ground_truth}\n[Context]\n{context}\n점수만 '점수: N' 형식으로."),
}

def _parse(raw):
    if not raw: return None
    m = re.search(r'점수\s*:\s*(\d)', raw)
    if m: return int(m.group(1))
    s = raw.strip()
    if s.isdigit() and 1 <= int(s) <= 5: return int(s)
    d = re.findall(r'\b[1-5]\b', raw); return int(d[0]) if d else None

async def _ask(prompt):
    for a in range(_RETRY):
        async with _SEM:
            try:
                r = await _client.chat.completions.create(
                    model=_M, messages=[{'role':'user','content':prompt}],
                    max_completion_tokens=512, timeout=60)
                return _parse(r.choices[0].message.content)
            except Exception:
                if a == _RETRY - 1: return None
                await asyncio.sleep(2**a)

async def score_one(q, ctx, ans, gt=None):
    tasks, none_keys = {}, []
    for m in ('faithfulness','relevance','rejection'):
        tasks[m] = _ask(_JP[m].format(context=ctx, query=q, answer=ans))
    for m in ('correctness','context_recall'):
        if gt and pd.notna(gt) and str(gt).strip():
            tasks[m] = _ask(_JP[m].format(ground_truth=gt, answer=ans, context=ctx))
        else: none_keys.append(m)
    tasks['context_precision'] = _ask(_JP['context_precision'].format(query=q, context=ctx))
    vals = await asyncio.gather(*tasks.values())
    res = dict(zip(tasks.keys(), vals))
    for k in none_keys: res[k] = None
    return res

_MET = ['faithfulness','relevance','rejection','correctness','context_precision','context_recall']

async def process_row(row):
    ans = row['answer']
    base = {'id':row['id'],'question':row['question'],'type':row['type'],'difficulty':row['difficulty']}
    if not isinstance(ans, str) or '오류' in str(ans)[:30]:
        for m in _MET: base[m] = None
    else:
        scores = await score_one(row['question'], row['retrieved_context'], ans, row.get('ground_truth_answer'))
        base.update(scores)
    return base

async def run_judge():
    gdf = pd.read_csv(GEN_PATH).drop_duplicates(subset='question')
    done, rows = set(), []
    if os.path.exists(JUDGE_PATH):
        ck = pd.read_csv(JUDGE_PATH)
        if 'relevance' in ck.columns and ck['relevance'].notna().any():
            ck = ck[ck['relevance'].notna()].drop_duplicates(subset='question')
            rows = ck.to_dict('records'); done = set(ck['question'])
            print('체크포인트 완료분:', len(done))
        else:
            print('기존 judge CSV 유효점수 없음 — 처음부터')
    pending = gdf[~gdf['question'].isin(done)]
    print('신규 평가 대상:', len(pending))

    chunk_size = 25   # ★ 끊김 대비 25행마다 저장
    recs = pending.to_dict('records')
    for i in range(0, len(recs), chunk_size):
        chunk = recs[i:i+chunk_size]
        res = await asyncio.gather(*[process_row(r) for r in chunk])
        rows.extend(res)
        pd.DataFrame(rows).to_csv(JUDGE_PATH, index=False, encoding='utf-8-sig')
        print(f'진행 {min(i+chunk_size,len(recs))} / {len(recs)}')

    out = pd.DataFrame(rows).drop_duplicates(subset='question')
    out.to_csv(JUDGE_PATH, index=False, encoding='utf-8-sig')
    print('✅ judge 완료:', len(out))
    print('점수 평균:', out[_MET].mean().round(3).to_dict())   # NaN 즉시 감지
    return out

judge_df = asyncio.get_event_loop().run_until_complete(run_judge())


In [ ]:
# [J2] Generation 요약
import pandas as pd
_MET=['faithfulness','relevance','rejection','correctness','context_precision','context_recall']
judge_df = pd.read_csv(JUDGE_PATH)
print('전체:', judge_df[_MET].mean().round(3).to_dict())
print('\n타입별:\n', judge_df.groupby('type')[_MET].mean().round(3))
s = judge_df.groupby('type')[_MET].mean()
s.loc['ALL'] = judge_df[_MET].mean()
s.round(3).to_csv(SUMM_PATH, encoding='utf-8-sig')
print('✅ 저장:', SUMM_PATH)
